# 🖋️ Train Triplet Network (Verification Candidate B)
### Metric Learning with Triplet Margin Loss
**License:** Apache 2.0 / BSD-3-Clause (Zero Ultralytics, Zero AGPL)


## Step 1: Check GPU Runtime


In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
!nvidia-smi


import os, shutil, getpass

# Clean previous directory if any
if os.path.exists("sign-detect-verify"):
    shutil.rmtree("sign-detect-verify")

# Check for GITHUB_TOKEN in Colab Secrets
github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

repo_url = "https://github.com/9m2usman/sign-detect-verify.git"
if github_token:
    ret = os.system(f"git clone https://{github_token}@github.com/9m2usman/sign-detect-verify.git")
else:
    ret = os.system(f"git clone {repo_url}")

if ret != 0 or not os.path.exists("sign-detect-verify"):
    print("\n[!] Public clone failed (repo is private or network restricted).")
    print("    Tip: Set repo to Public in GitHub Settings OR enter your GitHub Token below:")
    token = getpass.getpass("GitHub Token (or press Enter if made public): ").strip()
    if token:
        os.system(f"git clone https://{token}@github.com/9m2usman/sign-detect-verify.git")
    else:
        os.system(f"git clone {repo_url}")

if os.path.exists("sign-detect-verify"):
    %cd sign-detect-verify
    !pip install -q -r requirements.txt
    !python -c "import ultralytics" 2>/dev/null && echo "WARNING: AGPL Detected!" || echo "[✓] Clean Permissive Environment: Zero AGPL."
else:
    raise RuntimeError("Failed to clone repository. Please make the repository public or provide a valid GitHub token.")


In [ ]:
!git clone https://github.com/9m2usman/sign-detect-verify.git
%cd sign-detect-verify
!pip install -q -r requirements.txt


## Step 3: Train Triplet Network with Triplet Margin Loss


In [ ]:
from verification.models.triplet_network import TripletSignatureNetwork, TripletLoss
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TripletSignatureNetwork(embedding_dim=256, pretrained=True).to(device)
criterion = TripletLoss(margin=0.5)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("[+] Triplet Network initialized on:", device)
print("    Model parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
